In [1]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
# # Text data
# from langchain_community.document_loaders.text import TextLoader

# loader = TextLoader("data/Python.txt", encoding="utf-8")

# document = loader.load()
# # document

In [4]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# # document

# Ingestion Pipeline

In [5]:
# Data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

C:\Users\Syed khizer\AppData\Local\Temp\ipykernel_22304\451959863.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader


# Data => Langchain Documents

In [6]:
def load_all_pdfs():
    folder_path="data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)
            
            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1

    print("Total pdf: ", num_docs)
    print("Total pages: ", len(all_docs))
    return all_docs

In [7]:
all_pdf_documents = load_all_pdfs()

Total pdf:  2
Total pages:  32


# Chunks

In [8]:
# chunks
# !pip install langchain_text_splitters

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [10]:
chunks = split_docs(all_pdf_documents)
len(chunks)

320

# Embeddings 

In [11]:
from sentence_transformers import SentenceTransformer

In [12]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        print("Loading model....",self.model_name)
        self.model=SentenceTransformer(self.model_name)
        print("Embedding dimensions= ",self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embedding shape: ",embeddings.shape)
        return embeddings

In [13]:
embedding_manager = EmbeddingManager()

Loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimensions=  384


C:\Users\Syed khizer\AppData\Local\Temp\ipykernel_22304\3203930644.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimensions= ",self.model.get_sentence_embedding_dimension())


# Vector Store

In [14]:
import chromadb
import uuid

In [21]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        # Create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        #create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("Initialized the vector store with collection: ", self.collection_name)
        print("Docs in collection: ", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Number of Docs does not match number of embedings")

        # store => ids, embedding, document, metadata 
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("Total documents added in vector store: ",len(documents_content))
        print("Documents in collection: ",self.collection.count())

In [22]:
vector_store = VectorStoreManager()

Initialized the vector store with collection:  pdf_documents
Docs in collection:  0


In [23]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

embedding shape:  (320, 384)
Total documents added in vector store:  320
Documents in collection:  320


# Retrieval Pipeline

In [24]:
from sklearn.metrics.pairwise import cosine_similarity

In [40]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents  = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })

            print(f"Retrieved {len(retrieved_docs)} documents")

        else:
            print("No doc found!")

        return retrieved_docs

In [41]:
rag_retiever = RAGRetriever(embedding_manager, vector_store)

In [49]:
rag_retiever.retrieve("What is decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape:  (1, 384)
Retrieved 1 documents


[{'id': 'doc_404b4fe4-f485-4b16-9ad3-851e0b74a389',
  'document': 'layers, produce outputs of dimensiondmodel = 512.\nDecoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head\nattention over the output of the encoder stack. Similar to the encoder, we employ residual connections\naround each of the sub-layers, followed by layer normalization. We also modify the self-attention',
  'metadata': {'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin',
   'firstpage': '5998',
   'total_pages': 11,
   'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. Fergus and S. Vishwanathan and R. Garnett',
   'created': '2017',
   'subject': 'Neural Information Processing Systems http://nips.cc/',
   'type': 'Conference Proceedings',
   'published': '2017',
   'produ